# ⚡ Notebook 04 — Delta Lake Optimization

**Goal:** Use `OPTIMIZE`, `Z-ORDER`, `VACUUM`, and table statistics to dramatically speed up queries on large Delta Tables.

> **Run time:** ~8 min

## Why Optimize?
```
Unoptimized Delta Table:           Optimized Delta Table:
  1,000 small Parquet files         10 large Parquet files
  Every query scans all files       Z-ORDER: skip irrelevant files
  Slow query: 45 seconds            Fast query: 0.3 seconds
  Disk: 2GB with overhead           Disk: 600MB compacted
```

In [ ]:
# Import Delta Lake metadata helpers and Spark functions used in the optimization walkthrough
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Open the transactions Delta table so its storage metadata can be inspected
dt_txn = DeltaTable.forName(spark, 'fact_transactions')

# Show the current file count and table size before running OPTIMIZE and maintenance commands
detail = dt_txn.detail()
detail.select('name','numFiles','sizeInBytes','partitionColumns').show(truncate=False)
print('Run DESCRIBE DETAIL for full metadata:')

## Step 1 — DESCRIBE DETAIL (Before Optimization)

In [ ]:
%%sql
-- Inspect the full Delta metadata for fact_transactions before optimization work begins
DESCRIBE DETAIL fact_transactions

## Step 2 — OPTIMIZE: File Compaction

Combines many small Parquet files into fewer large files. Reduces I/O overhead.

In [ ]:
%%sql
-- Compact the fact_transactions table into fewer, larger files to reduce small-file overhead
-- Compact small files into larger ones (target: 128MB per file)
OPTIMIZE fact_transactions

## Step 3 — Z-ORDER: Data Layout Optimization

Z-ORDER physically co-locates related data in the same files. Queries on the Z-ORDER columns skip entire files instead of scanning everything.

In [ ]:
%%sql
-- Re-cluster fact_transactions by CustomerID and TransactionDate to improve selective query pruning
-- Z-ORDER by the most common JOIN/filter keys
-- After this, queries filtering on CustomerID or TransactionDate
-- will skip ~90% of files
OPTIMIZE fact_transactions
ZORDER BY (CustomerID, TransactionDate)

In [ ]:
%%sql
-- Apply the same Z-ORDER maintenance pattern to the loans fact table
OPTIMIZE fact_loans
ZORDER BY (CustomerID, LoanStatus)

## Step 4 — ANALYZE TABLE: Update Statistics

Updates column-level statistics so Spark's query optimizer makes better decisions.

In [ ]:
%%sql
-- Refresh column statistics on fact_transactions so the optimizer has up-to-date metadata
ANALYZE TABLE fact_transactions COMPUTE STATISTICS FOR ALL COLUMNS

In [ ]:
%%sql
-- Refresh column statistics on fact_loans after optimization
ANALYZE TABLE fact_loans COMPUTE STATISTICS FOR ALL COLUMNS

## Step 5 — VACUUM: Remove Old Files

Delta keeps old file versions for time travel (default: 7 days). VACUUM deletes files older than the retention period to reclaim disk space.

In [ ]:
%%sql
-- Check recent transaction history to see the versions created before vacuum maintenance
-- Check how many versions exist before vacuum
DESCRIBE HISTORY fact_transactions LIMIT 10

In [ ]:
# Disable the demo safety check so a zero-hour dry run can preview which files vacuum would remove
# VACUUM removes files older than retention period
# Default = 7 days. We use 0 hours here for demo only (NOT for production!)
# In production: VACUUM fact_transactions RETAIN 168 HOURS (7 days)
spark.sql('SET spark.databricks.delta.retentionDurationCheck.enabled = false')

# Run VACUUM in dry-run mode so the notebook previews deletable files without removing anything
spark.sql('VACUUM fact_transactions RETAIN 0 HOURS DRY RUN')  # DRY RUN = preview only

## Step 6 — DESCRIBE DETAIL (After Optimization)

In [ ]:
%%sql
-- Re-check the table metadata after optimization and maintenance commands complete
DESCRIBE DETAIL fact_transactions

## Step 7 — Benchmark Query (Before vs After)

In [ ]:
# Import timing utilities so the query can be benchmarked after Z-ORDER optimization
import time

# Define a customer-level aggregation query that benefits from clustering on CustomerID
query = """
    SELECT CustomerID, SUM(Amount) AS TotalSpend, COUNT(*) AS TxnCount
    FROM fact_transactions
    WHERE CustomerID IN ('CUST0001','CUST0050','CUST0100','CUST0200','CUST0300')
    GROUP BY CustomerID
"""

# Start a timer and execute the filtered aggregation query against the optimized table
start = time.time()
result = spark.sql(query)
result.show()
elapsed = time.time() - start

# Print the elapsed runtime so the optimization benefit can be compared to a pre-optimization baseline
print(f'
Query time after Z-ORDER optimization: {elapsed:.2f}s')
print('(Compare with full scan before optimization to see improvement)')